In [1]:
from pathlib import Path
import importlib
import os
import uproot
import polars as pl
import plotly.graph_objects as go
import qmirt_utility as qmirt
import numpy as np

importlib.reload(qmirt)

<module 'qmirt_utility' from '/home/fanghan/Work/RPIL/QMIRT/gate10mc/dev/python/qmirt-utility/qmirt_utility/__init__.py'>

In [2]:
style_sheet_path = "wrl_stylesheet.example.json"

fig = qmirt.plot.plot_wrl_file(
    "../../simple_lyso_sipm.wrl",
    style_sheet=style_sheet_path,
    exclude_patterns=["world:*"],
    # exclude_patterns=["crystal:*", "world:*"],
    exclude_mode="replace",
)
fig.show()

In [3]:
output_dir = os.path.abspath(os.path.join(os.getcwd(), "../../output"))
root_filename = "lyso_sipm_phsa_output.root"
with uproot.open(Path(output_dir) / root_filename) as file:
    tree = file["crystal_phsa"]
    crystal_phsa_df =  pl.DataFrame(tree.arrays(tree.keys(), library="np"))
    tree = file["sipm_phsa"]
    sipm_phsa_df = pl.DataFrame(tree.arrays(tree.keys(), library="np"))
    tree = file["opt_phsa"]
    opt_phsa_df = pl.DataFrame(tree.arrays(tree.keys(), library="np"))
print(crystal_phsa_df.shape)
print(opt_phsa_df.shape)
print(sipm_phsa_df.shape)


(668332, 16)
(34806, 16)
(8, 16)


In [5]:
# Sort the dataframes by EventID and TrackID
crystal_phsa_df = crystal_phsa_df.sort(["EventID", "TrackID","CurrentStepNumber"])
print(crystal_phsa_df.head())

shape: (5, 16)
┌─────────┬──────────┬─────────┬─────────────┬───┬─────────────┬────────────┬────────────┬─────────┐
│ EventID ┆ ParentID ┆ TrackID ┆ CurrentStep ┆ … ┆ TrackCreato ┆ EventKinet ┆ KineticEne ┆ PDGCode │
│ ---     ┆ ---      ┆ ---     ┆ Number      ┆   ┆ rProcess    ┆ icEnergy   ┆ rgy        ┆ ---     │
│ i32     ┆ i32      ┆ i32     ┆ ---         ┆   ┆ ---         ┆ ---        ┆ ---        ┆ i32     │
│         ┆          ┆         ┆ i32         ┆   ┆ str         ┆ f64        ┆ f64        ┆         │
╞═════════╪══════════╪═════════╪═════════════╪═══╪═════════════╪════════════╪════════════╪═════════╡
│ 0       ┆ 0        ┆ 1       ┆ 2           ┆ … ┆ none        ┆ 0.511      ┆ 0.511      ┆ 22      │
│ 0       ┆ 0        ┆ 1       ┆ 3           ┆ … ┆ none        ┆ 0.511      ┆ 0.338498   ┆ 22      │
│ 0       ┆ 1        ┆ 2       ┆ 1           ┆ … ┆ Scintillati ┆ 0.511      ┆ 0.000003   ┆ -22     │
│         ┆          ┆         ┆             ┆   ┆ on          ┆            

In [4]:
# Get Unique event IDs in the crystal_phsa_df
event_ids = crystal_phsa_df["EventID"].unique()
# For each event ID, count the number of unique track IDs in the crystal_phsa_df
for event_id in event_ids:
    track_ids = crystal_phsa_df.filter(
        (pl.col("EventID") == event_id) & (pl.col("TrackID") != 1))["TrackID"].to_numpy()
    if len(track_ids) == 0:
        print(f"Event ID: {event_id}\n Number of unique tracks: 0\n")
        continue
    unique_track_ids, step_counts = np.unique(track_ids, return_counts=True)

    # Count the unique TrackIDs and exclude TrackID == 1
    # which is the primary particle track
    print(f"Event ID: {event_id}\n Number of unique tracks: {len(unique_track_ids)}\n" +
          f"  Avg N steps per track: {np.mean(step_counts):.2f}\n" +
          f"  Max N steps per track: {np.max(step_counts)}\n" +
          f"  Min N steps per track: {np.min(step_counts)}\n")




Event ID: 0
 Number of unique tracks: 5136
  Avg N steps per track: 9.34
  Max N steps per track: 98
  Min N steps per track: 1

Event ID: 10000
 Number of unique tracks: 2336
  Avg N steps per track: 9.35
  Max N steps per track: 88
  Min N steps per track: 1

Event ID: 20000
 Number of unique tracks: 7705
  Avg N steps per track: 11.03
  Max N steps per track: 104
  Min N steps per track: 1

Event ID: 30000
 Number of unique tracks: 0

Event ID: 40000
 Number of unique tracks: 15116
  Avg N steps per track: 9.55
  Max N steps per track: 107
  Min N steps per track: 1

Event ID: 50000
 Number of unique tracks: 0

Event ID: 60000
 Number of unique tracks: 0

Event ID: 70000
 Number of unique tracks: 0

Event ID: 80000
 Number of unique tracks: 0

Event ID: 90000
 Number of unique tracks: 0

Event ID: 100000
 Number of unique tracks: 12841
  Avg N steps per track: 10.99
  Max N steps per track: 104
  Min N steps per track: 1

Event ID: 110000
 Number of unique tracks: 8966
  Avg N steps

In [5]:
crystal_event_phsa_df=crystal_phsa_df.filter(pl.col("EventID") == 0).sort("ParentID")
track_creator = crystal_event_phsa_df["TrackCreatorProcess"].to_numpy()
unique_track_creators, track_creator_counts = np.unique(track_creator, return_counts=True)
print(f"Unique TrackCreatorProcess: {unique_track_creators}\nCounts: {track_creator_counts}")


Unique TrackCreatorProcess: ['Scintillation' 'none']
Counts: [47953     2]


In [4]:
#Get the unique TrackIDs and their counts for the event with EventID == 0
track_ids_crystal = crystal_phsa_df.filter(pl.col("EventID") == 0).select(pl.col("TrackID")).to_numpy()
track_ids_epoxy = opt_phsa_df.filter(pl.col("EventID") == 0).select(pl.col("TrackID")).to_numpy()
unique_track_ids_crystal, track_id_counts_crystal = np.unique(track_ids_crystal, return_counts=True)
unique_track_ids_epoxy, track_id_counts_epoxy = np.unique(track_ids_epoxy, return_counts=True)
# Print number of unique TrackIDs in crystal_phsa_df and opt_phsa_df for EventID == 0
print(f"Number of unique TrackIDs in crystal_phsa_df for EventID == 0: {len(unique_track_ids_crystal)}\n" +
      f"Number of unique TrackIDs in opt_phsa_df for EventID == 0: {len(unique_track_ids_epoxy)}\n")

Number of unique TrackIDs in crystal_phsa_df for EventID == 0: 5137
Number of unique TrackIDs in opt_phsa_df for EventID == 0: 2371



In [11]:
# Check if unique track IDs in epoxy are a subset of unique track IDs in crystal
is_subset = set(unique_track_ids_epoxy).issubset(set(unique_track_ids_crystal))
# print out what track IDs in epoxy are not in crystal if not subset
if not is_subset:
    epoxy_not_in_crystal = set(unique_track_ids_epoxy) - set(unique_track_ids_crystal)
    # Print out the length of the set and the track IDs
    print(f"Number of unique TrackIDs in opt_phsa_df that are not in crystal_phsa_df for EventID == 0: {len(epoxy_not_in_crystal)}\n" +
          f"TrackIDs in opt_phsa_df that are not in crystal_phsa_df for EventID == 0: {epoxy_not_in_crystal}\n")
else:
    print("All unique TrackIDs in opt_phsa_df for EventID == 0 are also in crystal_phsa_df for EventID == 0\n")

# Get the TrackCreatorProcess for the tracks in the opt_phsa_df but not in the crystal_phsa_df for EventID == 0
if not is_subset:
    track_creator_epoxy_not_in_crystal = opt_phsa_df.filter(pl.col("EventID") == 0).filter(pl.col("TrackID").is_in(epoxy_not_in_crystal))["TrackCreatorProcess"].to_numpy()
    unique_track_creators_epoxy_not_in_crystal, track_creator_counts_epoxy_not_in_crystal = np.unique(track_creator_epoxy_not_in_crystal, return_counts=True)
    print(f"Unique TrackCreatorProcess for tracks in opt_phsa_df but not in crystal_phsa_df for EventID == 0: {unique_track_creators_epoxy_not_in_crystal}\n" +
          f"Counts: {track_creator_counts_epoxy_not_in_crystal}\n")
    # Also get the ParentID for these tracks
    parent_id_epoxy_not_in_crystal = opt_phsa_df.filter(pl.col("EventID") == 0).filter(pl.col("TrackID").is_in(epoxy_not_in_crystal))["ParentID"].to_numpy()
    unique_parent_id_epoxy_not_in_crystal, parent_id_counts_epoxy_not_in_crystal = np.unique(parent_id_epoxy_not_in_crystal, return_counts=True)
    print(f"Unique ParentID for tracks in opt_phsa_df but not in crystal_phsa_df for EventID == 0: {unique_parent_id_epoxy_not_in_crystal}\n" +
          f"Counts: {parent_id_counts_epoxy_not_in_crystal}\n")

Number of unique TrackIDs in opt_phsa_df that are not in crystal_phsa_df for EventID == 0: 35
TrackIDs in opt_phsa_df that are not in crystal_phsa_df for EventID == 0: {np.int32(2819), np.int32(3716), np.int32(5), np.int32(1669), np.int32(3720), np.int32(3339), np.int32(5266), np.int32(1302), np.int32(3752), np.int32(3629), np.int32(48), np.int32(304), np.int32(5171), np.int32(1972), np.int32(955), np.int32(1085), np.int32(573), np.int32(1090), np.int32(2755), np.int32(836), np.int32(716), np.int32(4184), np.int32(985), np.int32(1121), np.int32(3425), np.int32(2275), np.int32(484), np.int32(2532), np.int32(5347), np.int32(5097), np.int32(4717), np.int32(2416), np.int32(2164), np.int32(4861), np.int32(1278)}

Unique TrackCreatorProcess for tracks in opt_phsa_df but not in crystal_phsa_df for EventID == 0: ['Scintillation']
Counts: [36]

Unique ParentID for tracks in opt_phsa_df but not in crystal_phsa_df for EventID == 0: [   1  629 2692]
Counts: [34  1  1]



In [19]:
# Get unique event IDs
event_ids, counts = np.unique(opt_phsa_df["EventID"].to_numpy(), return_counts=True)
aligned_data = [[event_id, count] for event_id, count in zip(event_ids, counts)]
qmirt.utils.print_list_aligned(aligned_data)

┌─────────────┐
│ 0      │ 25 │
│ 10000  │ 86 │
│ 20000  │ 53 │
│ 30000  │ 1  │
│ 40000  │ 27 │
│ 60000  │ 1  │
│ 70000  │ 1  │
│ 80000  │ 1  │
│ 90000  │ 1  │
│ 100000 │ 88 │
│ 110000 │ 46 │
│ 120000 │ 1  │
│ 130000 │ 1  │
│ 140000 │ 69 │
│ 150000 │ 1  │
└─────────────┘


In [17]:
# Get unique event IDs
event_ids, counts = np.unique(sipm_phsa_df["EventID"].to_numpy(), return_counts=True)
aligned_data = [[event_id, count] for event_id, count in zip(event_ids, counts)]
qmirt.utils.print_list_aligned(aligned_data)

┌────────────┐
│ 30000  │ 1 │
│ 60000  │ 1 │
│ 70000  │ 1 │
│ 80000  │ 1 │
│ 90000  │ 1 │
│ 120000 │ 1 │
│ 130000 │ 1 │
│ 150000 │ 1 │
└────────────┘


In [21]:
# Get track IDs for the EventID == 0 for both crystal_phsa_df and opt_phsa_df
event_id = 0
crystal_event_df = crystal_phsa_df.filter(pl.col("EventID") == event_id).sort("TrackID")
opt_event_df = opt_phsa_df.filter(pl.col("EventID") == event_id).sort("TrackID")
sipm_event_df = sipm_phsa_df.filter(pl.col("EventID") == event_id).sort("TrackID")
print(crystal_event_df.head())
print(opt_event_df.head())
print(sipm_event_df.head())

shape: (5, 16)
┌─────────┬──────────┬─────────┬─────────────┬───┬─────────────┬────────────┬────────────┬─────────┐
│ EventID ┆ ParentID ┆ TrackID ┆ CurrentStep ┆ … ┆ TrackCreato ┆ EventKinet ┆ KineticEne ┆ PDGCode │
│ ---     ┆ ---      ┆ ---     ┆ Number      ┆   ┆ rProcess    ┆ icEnergy   ┆ rgy        ┆ ---     │
│ i32     ┆ i32      ┆ i32     ┆ ---         ┆   ┆ ---         ┆ ---        ┆ ---        ┆ i32     │
│         ┆          ┆         ┆ i32         ┆   ┆ str         ┆ f64        ┆ f64        ┆         │
╞═════════╪══════════╪═════════╪═════════════╪═══╪═════════════╪════════════╪════════════╪═════════╡
│ 0       ┆ 0        ┆ 1       ┆ 2           ┆ … ┆ none        ┆ 0.511      ┆ 0.511      ┆ 22      │
│ 0       ┆ 0        ┆ 1       ┆ 3           ┆ … ┆ none        ┆ 0.511      ┆ 0.338498   ┆ 22      │
│ 0       ┆ 1        ┆ 2       ┆ 1           ┆ … ┆ Scintillati ┆ 0.511      ┆ 0.000003   ┆ -22     │
│         ┆          ┆         ┆             ┆   ┆ on          ┆            

In [12]:
# Get the track of the first event
event_id = event_ids[0]
event_df = crystal_phsa_df.filter(pl.col("EventID") == event_id).sort("TrackID")
unique_track_ids = event_df["TrackID"].unique()

fig = qmirt.plot.plot_wrl_file(
    "../../simple_lyso_sipm.wrl",
    style_sheet=style_sheet_path,
    exclude_patterns=["world:*"],
    # exclude_patterns=["crystal:*", "world:*"],
    exclude_mode="replace",
)

selected_track_ids = unique_track_ids[:5]  # Select the first 5 tracks for plotting
# Add tracks to the figure
for track_id in selected_track_ids:
    track_df = event_df.filter(pl.col("TrackID") == track_id)
    fig.add_trace(
        qmirt.plot.track.create_plotly_go_track(
            track_df,
            style={
                "marker": {"size": 2, "color": "blue"},
                "line": {"width": 2, "color": "blue"},
            },
        )
    )


fig.show()

In [8]:
# Check if the EventIDs in crystal_phsa_df and opt_phsa_df match
all_event_ids_match = np.all(
    crystal_phsa_df["EventID"].unique().to_numpy()
    == opt_phsa_df["EventID"].unique().to_numpy()
)
print("All EventIDs match" if all_event_ids_match else "EventIDs do not match")

All EventIDs match


In [9]:
## Pick EventID = 0 for visualization
event_id_to_visualize = 0
crystal_event_df = crystal_phsa_df.filter(pl.col("EventID") == event_id_to_visualize)
opt_event_df = opt_phsa_df.filter(pl.col("EventID") == event_id_to_visualize)
print(crystal_event_df.shape)
print(opt_event_df.shape)

## Get number of tracks in this event
print(
    f"Number of tracks in crystal for EventID {event_id_to_visualize}: {crystal_event_df['TrackID'].n_unique()}"

)
print(f"Number of tracks in optical photon detector for EventID {event_id_to_visualize}: {opt_event_df['TrackID'].n_unique()}")


(22848, 16)
(13, 16)
Number of tracks in crystal for EventID 0: 1225
Number of tracks in optical photon detector for EventID 0: 13


In [10]:
## Print unique track IDs in opt_event_df
print(
    f"Unique TrackIDs in opt_event_df for EventID {
        event_id_to_visualize}: {opt_event_df['TrackID'].unique().to_numpy()}"
)
## Print unique parent IDs in opt_event_df
print(
    f"Unique ParentIDs in opt_event_df for EventID {event_id_to_visualize}: {opt_event_df['ParentID'].unique().to_numpy()}"
)


Unique TrackIDs in opt_event_df for EventID 0: [1217 1219 1220 1222 1223 1224 1226 1227 1229 1230 1236 1237 1240]
Unique ParentIDs in opt_event_df for EventID 0: [  60  223  293  535  614  627  652  724  765  849  980 1031 1176]


In [11]:
track_global_id_crystal = crystal_phsa_df['EventID'].to_numpy().copy().astype(np.uint64)
np.multiply(track_global_id_crystal, 100000, out=track_global_id_crystal)
np.add(track_global_id_crystal, crystal_phsa_df['TrackID'].to_numpy().astype(np.uint64), out=track_global_id_crystal)
print(f"Track Global IDs in crystal_phsa_df:\n  shape: {track_global_id_crystal.shape}")

track_global_id_opt = opt_phsa_df['EventID'].to_numpy().copy().astype(np.uint64)
np.multiply(track_global_id_opt, 100000, out=track_global_id_opt)
np.add(track_global_id_opt, opt_phsa_df['ParentID'].to_numpy().astype(np.uint64), out=track_global_id_opt)
print(f"Track Global IDs in opt_phsa_df:\n  shape: {track_global_id_opt.shape}")

passed_through_track_ids = np.intersect1d(track_global_id_crystal, track_global_id_opt)
print(f"Track IDs that passed through both crystal and optical photon detector: {passed_through_track_ids.shape[0]}")

# Get the Event ID correspoding to non-passed through tracks
non_passed_through_track_ids = np.setdiff1d( track_global_id_opt, track_global_id_crystal)
print(non_passed_through_track_ids//100000)


Track Global IDs in crystal_phsa_df:
  shape: (346350,)
Track Global IDs in opt_phsa_df:
  shape: (160,)
Track IDs that passed through both crystal and optical photon detector: 158
[140000]


In [20]:
## Get the steps in the optical photon detector that correspond to the non-passed through tracks
non_passed_event_ids = non_passed_through_track_ids // 100000
non_passed_parent_ids = non_passed_through_track_ids % 100000
print(f"Not passed through track Event IDs: {non_passed_event_ids}")
print(f"Not passed through track Parent IDs: {non_passed_parent_ids}")
non_passed_steps_opt = opt_phsa_df.filter(opt_phsa_df['EventID']==non_passed_event_ids[0], opt_phsa_df['ParentID']==non_passed_parent_ids[0])
print(f"Steps in optical photon detector corresponding to non-passed through tracks:\n  shape: {non_passed_steps_opt.shape}")
# Print the TrackCreatorProcess for the non-passed through tracks
print(f"TrackCreatorProcess for non-passed through tracks: {non_passed_steps_opt['TrackCreatorProcess'].unique().to_numpy()}")

Not passed through track Event IDs: [140000]
Not passed through track Parent IDs: [456]
Steps in optical photon detector corresponding to non-passed through tracks:
  shape: (1, 16)
TrackCreatorProcess for non-passed through tracks: ['Scintillation']


In [13]:
style_sheet_path = "wrl_stylesheet.example.json"

fig = plot_wrl.plot_wrl_file(
    "../../sim_geometry.wrl",
    style_sheet=style_sheet_path,
    exclude_patterns=["world:*"],
    # exclude_patterns=["crystal:*", "world:*"],
    exclude_mode="replace",
)


opt_event_df = opt_phsa_df.filter(pl.col("EventID")== non_passed_through_track_ids//100000)
crystal_event_df = crystal_phsa_df.filter(pl.col("EventID") == non_passed_through_track_ids//100000)

# selected_track_id = np.arange(0,4)
colors = ["red", "blue", "green", "orange"]
tracks_to_plot = []
opt_track_ids = opt_event_df["TrackID"].unique().to_numpy()
opt_track_parent_ids = opt_event_df["ParentID"].unique().to_numpy()
for i, track_id in enumerate(opt_track_parent_ids):
    tracks_to_plot.append(create_plotly_go_track(
        crystal_event_df.filter(pl.col("TrackID") == track_id),
        style={"line": {"color": colors[1], "width": 1}, "marker": {"color": colors[1], "size": 2}},
    ))

opt_track_ids = opt_event_df["TrackID"].unique().to_numpy()
for i, track_id in enumerate(opt_track_ids):
    tracks_to_plot.append(create_plotly_go_track_with_steps(
        opt_event_df.filter(pl.col("TrackID") == track_id),
        style={"line": {"color": colors[0], "width": 1}, "marker": {"color": colors[0], "size": 2}},
    ))
fig.add_traces(
    tracks_to_plot,
)
fig.show()